# PyTorch Neural Networks

**PyTorch Interview Preparation · Notebook 04 · Estimated study time: 2–3 hours**

The loop is: **read a requirement → build the module → predict shapes → run it → inspect parameters → explain the architecture**.


## 1. Setup

This notebook uses only PyTorch and runs on CPU.


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)
print("PyTorch version:",torch.__version__)

def count_parameters(model):
    total=sum(p.numel() for p in model.parameters())
    trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total,trainable


## 2. What Is `torch.nn`?

`torch.nn` provides neural-network building blocks: layers, activations, normalization, regularization, and the base class used to compose models. Examples include `nn.Linear`, `nn.ReLU`, `nn.Conv2d`, and `nn.Dropout`.


## 3. `nn.Module` — 🔥 Interview Essential

`nn.Module` is the base class for PyTorch models and layers.

```text
class Model(nn.Module) → inherit module behavior
__init__()             → define/register components
super().__init__()     → initialize Module machinery
forward(x)             → define data flow
```


In [ ]:
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self,x):
        return x

simple=SimpleModel()
print(simple(torch.tensor([1.,2.])))


### Why `super().__init__()`? — 🔥 Interview Essential

It initializes the `nn.Module` base machinery so PyTorch can register parameters, child modules, buffers, and hooks correctly.


## 4. Building Your First Model


In [ ]:
class LinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear=nn.Linear(3,1)
    def forward(self,x):
        return self.linear(x)

model=LinearModel()
x=torch.randn(5,3)
output=model(x)
print("(5,3) -> Linear(3,1) ->",tuple(output.shape))


### 🟢 Easy — First custom model

Define `MyModel` with one `nn.Linear(4,2)` and a working forward pass.


In [ ]:
# Exercise: Define `MyModel` with one `nn.Linear(4,2)` and a working forward pass.
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO
    def forward(self,x):
        # TODO
        pass

model=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if model is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(model,nn.Module), "Instantiate MyModel as model"
    x=torch.randn(8,4)
    output=model(x)
    assert output.shape==(8,2), "Expected output shape (8,2)"
    assert any(isinstance(m,nn.Linear) and m.in_features==4 and m.out_features==2 for m in model.modules()), "Register Linear(4,2)"
    print("✅ Correct!")


## 5. Understanding `__init__` and `forward`

Define reusable layers in `__init__`; apply them in `forward`. Registered components appear in parameter/module traversal and move with the model between devices.


## 6. Calling `model(x)` — 🔥 Interview Essential

`forward()` defines computation, but call `model(x)`. That invokes `nn.Module.__call__()`, which runs module machinery such as hooks before/after calling `forward()`.


## 7. `nn.Linear` — 🔥 Interview Essential

`nn.Linear(in_features,out_features)` applies `y=xWᵀ+b` to the final feature dimension.


In [ ]:
layer=nn.Linear(10,4)
x=torch.randn(32,10)
y=layer(x)
print("output:",y.shape,"weight:",layer.weight.shape,"bias:",layer.bias.shape)


For `Linear(10,4)`: input `(batch,10)` → output `(batch,4)`, weight `(4,10)`, bias `(4,)`.


### 🟢 Easy — Linear layer

Create `answer=nn.Linear(12,5)`.


In [ ]:
# Exercise: Create `answer=nn.Linear(12,5)`.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.Linear), "Create an nn.Linear layer"
    assert answer.in_features==12 and answer.out_features==5, "Check feature sizes"
    assert answer(torch.randn(7,12)).shape==(7,5), "Output should preserve batch and replace features"
    print("✅ Correct!")


### 🟢 Easy — Parameter shapes

For `layer=nn.Linear(128,64)`, assign `(weight_shape,bias_shape)` to `answer`.


In [ ]:
# Exercise: For `layer=nn.Linear(128,64)`, assign `(weight_shape,bias_shape)` to `answer`.
layer=nn.Linear(128,64)
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert answer==(torch.Size([64,128]),torch.Size([64])), "Linear weight is (out,in); bias is (out,)"
    print("✅ Correct!")


## 8. Parameter Shapes and Counts — 🔥 Interview Essential

For `Linear(in,out)`, total parameters = `out×in + out` when bias is enabled.


In [ ]:
layer=nn.Linear(10,4)
print("manual:",10*4+4,"programmatic:",sum(p.numel() for p in layer.parameters()))


### Eight parameter-count predictions

1. `Linear(10,4)` → ?
2. `Linear(100,50)` → ?
3. `Linear(50,10)` → ?
4. `Linear(128,64)` → ?
5. `Linear(64,1)` → ?
6. `Linear(784,128)` → ?
7. `Linear(32,32)` → ?
8. `Linear(5,3,bias=False)` → ?

<details><summary>Show answers</summary>

1. `44`
2. `5,050`
3. `510`
4. `8,256`
5. `65`
6. `100,480`
7. `1,056`
8. `15`

</details>


### 🟢 Easy — Count two layers

Manually calculate the total parameters in `Linear(100,50) -> Linear(50,10)` and assign the integer to `answer`.


In [ ]:
# Exercise: Manually calculate the total parameters in `Linear(100,50) -> Linear(50,10)` and assign the integer to `answer`.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    model=nn.Sequential(nn.Linear(100,50),nn.Linear(50,10))
    assert isinstance(answer,int), "Use an integer count"
    assert answer==sum(p.numel() for p in model.parameters()), "Include every weight and bias"
    assert answer==5560, "Recheck both layers"
    print("✅ Correct!")


## 9. Activation Functions — 🔥 Interview Essential

Activations introduce nonlinearity.

| Activation | Practical note |
|---|---|
| ReLU | efficient, common hidden activation; can have dead units |
| LeakyReLU | keeps a small negative slope |
| GELU | smooth and common in transformers |
| Sigmoid | maps to `(0,1)`, often binary probability-like output |
| Tanh | maps to `(-1,1)`, historically common in RNNs |
| Softmax | normalizes class scores along a chosen dimension |


In [ ]:
x=torch.tensor([-2.,0.,3.])
for activation in [nn.ReLU(),nn.LeakyReLU(.01),nn.GELU(),nn.Sigmoid(),nn.Tanh()]:
    print(type(activation).__name__,activation(x))


**Loss preview:** pass raw logits to `BCEWithLogitsLoss` (no manual sigmoid) and `CrossEntropyLoss` (no manual softmax); those losses include stable transformations internally.


In [ ]:
softmax=nn.Softmax(dim=1)
logits=torch.tensor([[1.,2.,3.],[2.,1.,0.]])
probs=softmax(logits)
print(probs)
print("row sums:",probs.sum(dim=1))


### 🟢 Easy — Activation stack

Create `answer` containing `Linear(6,8) -> GELU -> Linear(8,2)`.


In [ ]:
# Exercise: Create `answer` containing `Linear(6,8) -> GELU -> Linear(8,2)`.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.Module), "Build an nn.Module"
    modules=list(answer.modules())
    assert sum(isinstance(m,nn.Linear) for m in modules)==2, "Expected two Linear layers"
    assert any(isinstance(m,nn.GELU) for m in modules), "Include GELU"
    assert answer(torch.randn(4,6)).shape==(4,2), "Expected output (4,2)"
    print("✅ Correct!")


## 10. Why Nonlinearity Matters — 🔥 Interview Essential

Stacking linear layers without activations is still equivalent to one linear transformation. `Linear → ReLU/GELU → Linear` can represent nonlinear relationships.


## 11. Building Multi-Layer Networks


In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1=nn.Linear(10,32)
        self.relu=nn.ReLU()
        self.fc2=nn.Linear(32,2)
    def forward(self,x):
        return self.fc2(self.relu(self.fc1(x)))

model=MLP()
print("(64,10) -> (64,32) -> (64,32) ->",model(torch.randn(64,10)).shape)


### 🟡 Medium — Build a three-layer MLP

Build `model`: 20 → 64 → ReLU → 32 → ReLU → 5 logits.


In [ ]:
# Exercise: Build `model`: 20 → 64 → ReLU → 32 → ReLU → 5 logits.
model=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if model is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(model,nn.Module), "model must be an nn.Module"
    assert model(torch.randn(16,20)).shape==(16,5), "Expected output (16,5)"
    linears=[m for m in model.modules() if isinstance(m,nn.Linear)]
    assert len(linears)>=3, "Expected at least three Linear layers"
    assert [(m.in_features,m.out_features) for m in linears[:3]]==[(20,64),(64,32),(32,5)], "Check layer dimensions"
    assert sum(isinstance(m,nn.ReLU) for m in model.modules())>=2, "Include nonlinearities"
    print("✅ Correct!")


## 12. `nn.Sequential` — 🔥 Interview Essential

Sequential passes each output directly into the next layer. It is ideal for straight-line models; a custom module is better for branches, skip connections, multiple inputs, or custom flow.


In [ ]:
model=nn.Sequential(nn.Linear(10,32),nn.ReLU(),nn.Linear(32,2))
print(model)
print(model(torch.randn(4,10)).shape)


### 🟢 Easy — Convert to Sequential

Create the equivalent `10 → 20 → ReLU → 3` Sequential model.


In [ ]:
# Exercise: Create the equivalent `10 → 20 → ReLU → 3` Sequential model.
model=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if model is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(model,nn.Sequential), "Use nn.Sequential"
    assert isinstance(model[0],nn.Linear) and isinstance(model[1],nn.ReLU) and isinstance(model[2],nn.Linear), "Check module order"
    assert model(torch.randn(6,10)).shape==(6,3), "Expected output (6,3)"
    print("✅ Correct!")


## 13. Custom Modules

PyTorch encourages composing reusable modules.


In [ ]:
class LinearBlock(nn.Module):
    def __init__(self,in_features,out_features):
        super().__init__()
        self.linear=nn.Linear(in_features,out_features)
        self.relu=nn.ReLU()
    def forward(self,x):
        return self.relu(self.linear(x))

class BlockModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1=LinearBlock(10,32)
        self.block2=LinearBlock(32,16)
        self.output=nn.Linear(16,2)
    def forward(self,x):
        return self.output(self.block2(self.block1(x)))

print(BlockModel()(torch.randn(5,10)).shape)


### 🟡 Medium — Reusable block

Define `block` as a module performing Linear(7,4) then ReLU.


In [ ]:
# Exercise: Define `block` as a module performing Linear(7,4) then ReLU.
block=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if block is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(block,nn.Module), "block must be a module"
    assert block(torch.randn(3,7)).shape==(3,4), "Expected output (3,4)"
    assert any(isinstance(m,nn.Linear) for m in block.modules()) and any(isinstance(m,nn.ReLU) for m in block.modules()), "Include Linear and ReLU"
    print("✅ Correct!")


## 14. Model Parameters — 🔥 Interview Essential

`model.parameters()` yields parameter tensors; `named_parameters()` yields `(name,parameter)` pairs. Optimizers later update these tensors.


In [ ]:
model=nn.Sequential(nn.Linear(3,4),nn.ReLU(),nn.Linear(4,2))
print("parameter shapes:",[tuple(p.shape) for p in model.parameters()])
for name,param in model.named_parameters():
    print(name,tuple(param.shape))


### `children()` vs `modules()`

`children()` returns immediate child modules. `modules()` recursively traverses the module itself and all descendants.


In [ ]:
print("children:",[type(m).__name__ for m in model.children()])
print("modules:",[type(m).__name__ for m in model.modules()])


## 15. Counting Parameters

Total and trainable counts can differ after freezing.


In [ ]:
model=nn.Sequential(nn.Linear(10,8),nn.ReLU(),nn.Linear(8,2))
print("before:",count_parameters(model))
for p in model[0].parameters(): p.requires_grad=False
print("after freeze:",count_parameters(model))


### 🟢 Easy — Total vs trainable

Freeze the first Linear layer and assign `(total,trainable)` to `answer`.


In [ ]:
# Exercise: Freeze the first Linear layer and assign `(total,trainable)` to `answer`.
model=nn.Sequential(nn.Linear(5,4),nn.ReLU(),nn.Linear(4,2))
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert answer==count_parameters(model), "Use parameter numel counts"
    assert answer==(34,10), "Total includes frozen parameters; trainable does not"
    assert all(not p.requires_grad for p in model[0].parameters()), "Freeze the first layer"
    print("✅ Correct!")


## 16. Logits and Output Layers — 🔥 Interview Essential

Logits are raw unnormalized model scores. For `(batch,classes)`, each row contains class scores. During common training, return logits directly to `CrossEntropyLoss`; for binary classification, return one raw logit per sample to `BCEWithLogitsLoss`.

```text
binary:     (batch,1)
multiclass: (batch,num_classes)
logits → softmax/sigmoid → probabilities (usually for interpretation)
```


In [ ]:
model=nn.Linear(5,3)
logits=model(torch.randn(4,5))
print(logits.shape,"samples=4, class scores=3")


### 🟢 Easy — Output head

Create a multiclass output layer for hidden size 64 and 7 classes.


In [ ]:
# Exercise: Create a multiclass output layer for hidden size 64 and 7 classes.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.Linear), "Use a Linear output head"
    assert answer.in_features==64 and answer.out_features==7, "Map hidden features to class logits"
    assert answer(torch.randn(9,64)).shape==(9,7), "One logit per class per sample"
    print("✅ Correct!")


## 17. Dropout — 🔥 Interview Essential

During training, `Dropout(p)` randomly zeros activations and scales survivors to preserve expected magnitude. In evaluation it is disabled.


In [ ]:
torch.manual_seed(42)
dropout=nn.Dropout(.5)
x=torch.ones(12)
dropout.train(); print("train:",dropout(x))
dropout.eval(); print("eval:",dropout(x))


### 🟢 Easy — Dropout module

Create `answer=nn.Dropout(p=0.3)`.


In [ ]:
# Exercise: Create `answer=nn.Dropout(p=0.3)`.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.Dropout), "Create nn.Dropout"
    assert answer.p==0.3, "Use probability 0.3"
    answer.eval(); x=torch.ones(5)
    assert torch.equal(answer(x),x), "Dropout is disabled in eval mode"
    print("✅ Correct!")


## 18. Batch Normalization — 🔥 Interview Essential

`BatchNorm1d(features)` is common for MLP features; `BatchNorm2d(channels)` is common for CNN feature maps. BatchNorm learns scale/shift and tracks running statistics.


In [ ]:
bn1=nn.BatchNorm1d(10)
print(bn1(torch.randn(32,10)).shape)
bn2=nn.BatchNorm2d(16)
print(bn2(torch.randn(8,16,12,12)).shape)


### BatchNorm train/eval behavior — 🔥 Interview Essential

Training uses batch statistics and updates running statistics. Evaluation uses stored running statistics. This is one major reason `model.eval()` matters.


### 🟢 Easy — BatchNorm shapes

Create a `BatchNorm1d` layer for 64 features.


In [ ]:
# Exercise: Create a `BatchNorm1d` layer for 64 features.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.BatchNorm1d) and answer.num_features==64, "Match the feature dimension"
    assert answer(torch.randn(8,64)).shape==(8,64), "Normalization preserves shape"
    print("✅ Correct!")


## 19. Layer Normalization — 🔥 Interview Essential for modern AI

`LayerNorm(128)` normalizes the final feature dimension of size 128 independently for each sample/token. It is extremely common in transformers.


In [ ]:
ln=nn.LayerNorm(128)
x=torch.randn(32,20,128)
y=ln(x)
print(y.shape,"mean of first token:",y[0,0].mean().item())


### 🟢 Easy — Transformer normalization

Create `LayerNorm` for hidden size 768.


In [ ]:
# Exercise: Create `LayerNorm` for hidden size 768.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.LayerNorm), "Create nn.LayerNorm"
    assert tuple(answer.normalized_shape)==(768,), "Normalize hidden size 768"
    assert answer(torch.randn(2,5,768)).shape==(2,5,768), "Preserve shape"
    print("✅ Correct!")


## 20. BatchNorm vs LayerNorm — 🔥 Interview Essential

| Feature | BatchNorm | LayerNorm |
|---|---|---|
| Depends on batch statistics | yes | no |
| Meaningful train/eval statistics change | yes | usually no |
| Common in CNNs | very common | less common |
| Common in transformers | rare | very common |
| Small-batch sensitivity | higher | lower |

BatchNorm uses statistics across a batch/channel arrangement; LayerNorm normalizes features within each sample or token.


## 21. `nn.Flatten`

By default, `nn.Flatten()` preserves batch dimension 0 and flattens from dimension 1.


In [ ]:
flatten=nn.Flatten()
x=torch.randn(32,3,28,28)
print(flatten(x).shape)  # (32,2352)


### 🟢 Easy — Flatten images

Create a module that flattens `(batch,64,7,7)` to `(batch,3136)`.


In [ ]:
# Exercise: Create a module that flattens `(batch,64,7,7)` to `(batch,3136)`.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.Module), "Use a module"
    assert answer(torch.randn(5,64,7,7)).shape==(5,3136), "Preserve batch and flatten features"
    print("✅ Correct!")


## 22. Embedding Layers — 🔥 Interview Essential for NLP / LLM roles

`nn.Embedding(num_embeddings,embedding_dim)` maps integer IDs to learned dense vectors. Inputs normally use `torch.long`; the weight shape is `(num_embeddings,embedding_dim)`.


In [ ]:
embedding=nn.Embedding(10000,128)
tokens=torch.tensor([[5,20,8],[9,4,2]],dtype=torch.long)
out=embedding(tokens)
print(tokens.shape,"->",out.shape,"weight:",embedding.weight.shape)


### 🟢 Easy — Token embedding

Create an embedding for vocabulary 5000 and dimension 64.


In [ ]:
# Exercise: Create an embedding for vocabulary 5000 and dimension 64.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.Embedding), "Create nn.Embedding"
    assert answer.num_embeddings==5000 and answer.embedding_dim==64, "Check vocabulary and vector size"
    assert answer(torch.randint(0,5000,(4,7))).shape==(4,7,64), "Append embedding dimension"
    print("✅ Correct!")


### Embedding parameter counts

1. `Embedding(10000,128)` → ?
2. `Embedding(50000,768)` → ?
3. `Embedding(30000,256)` → ?
4. `Embedding(1000,32)` → ?

<details><summary>Show answers</summary>

1. `1,280,000`
2. `38,400,000`
3. `7,680,000`
4. `32,000`

</details>


## 23. Introduction to Convolution

`Conv1d` operates on `(N,C,L)` sequences; `Conv2d` operates on `(N,C,H,W)` images. This notebook focuses on basic Conv2d syntax and shapes.


In [ ]:
conv1=nn.Conv1d(8,16,kernel_size=3,padding=1)
print("Conv1d:",conv1(torch.randn(4,8,20)).shape)
conv2=nn.Conv2d(3,16,kernel_size=3)
print("Conv2d:",conv2(torch.randn(32,3,28,28)).shape)


### Conv2d output size — 🔥 Interview Useful

For dilation 1:

`floor((input + 2×padding - kernel_size) / stride) + 1`

`28,k=3,p=0,s=1 → 26`; `28,k=3,p=1,s=1 → 28`. Stride greater than one downsamples.


### Ten convolution-size predictions

1. input 28, k3, p0, s1 → ?
2. input 28, k3, p1, s1 → ?
3. input 32, k3, p1, s2 → ?
4. input 64, k5, p0, s1 → ?
5. input 64, k5, p2, s1 → ?
6. input 31, k3, p0, s2 → ?
7. input 224, k7, p3, s2 → ?
8. input 14, k1, p0, s1 → ?
9. input 10, k4, p0, s2 → ?
10. input 15, k3, p2, s1 → ?

<details><summary>Show answers</summary>

1. `26`
2. `28`
3. `16`
4. `60`
5. `64`
6. `15`
7. `112`
8. `14`
9. `4`
10. `17`

</details>


### 🟡 Medium — Same-size convolution

Create Conv2d from 3 to 32 channels, kernel 3, padding 1.


In [ ]:
# Exercise: Create Conv2d from 3 to 32 channels, kernel 3, padding 1.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.Conv2d), "Create nn.Conv2d"
    assert (answer.in_channels,answer.out_channels,answer.kernel_size,answer.padding)==(3,32,(3,3),(1,1)), "Check convolution arguments"
    assert answer(torch.randn(2,3,24,24)).shape==(2,32,24,24), "Spatial size should stay unchanged"
    print("✅ Correct!")


### Conv2d parameter count

Weights have shape `(out_channels,in_channels,kH,kW)` (ignoring groups here); bias has one value per output channel.


In [ ]:
conv=nn.Conv2d(3,16,3)
print(conv.weight.shape,conv.bias.shape,count_parameters(conv)[0])  # 16*3*3*3+16=448


### 🟡 Medium — Count convolution parameters

Calculate parameters for `Conv2d(16,32,3,bias=True)` and assign integer to `answer`.


In [ ]:
# Exercise: Calculate parameters for `Conv2d(16,32,3,bias=True)` and assign integer to `answer`.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    conv=nn.Conv2d(16,32,3)
    assert answer==sum(p.numel() for p in conv.parameters()), "Count kernel weights and output-channel biases"
    assert answer==4640, "Recheck out×in×k×k + out"
    print("✅ Correct!")


## 24. Pooling Layers

`MaxPool2d(2)` commonly halves height/width. `AdaptiveAvgPool2d((1,1))` converts any spatial resolution to a fixed 1×1 per channel.


In [ ]:
pool=nn.MaxPool2d(2)
print(pool(torch.randn(32,16,28,28)).shape)
adaptive=nn.AdaptiveAvgPool2d((1,1))
print(adaptive(torch.randn(32,128,7,7)).shape)


### 🟢 Easy — Adaptive classifier pool

Create a module mapping any `(N,128,H,W)` to `(N,128,1,1)`.


In [ ]:
# Exercise: Create a module mapping any `(N,128,H,W)` to `(N,128,1,1)`.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.AdaptiveAvgPool2d), "Use adaptive average pooling"
    assert answer(torch.randn(3,128,9,13)).shape==(3,128,1,1), "Output should not depend on input H,W"
    print("✅ Correct!")


## 25. Shape Reasoning Through Networks — 🔥 Interview Essential

### A simple CNN shape flow


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features=nn.Sequential(
            nn.Conv2d(3,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.classifier=nn.Sequential(nn.Flatten(),nn.Linear(32*8*8,10))
    def forward(self,x):
        return self.classifier(self.features(x))

cnn=SimpleCNN()
print(cnn(torch.randn(8,3,32,32)).shape)


```text
(8,3,32,32) → Conv → (8,16,32,32) → Pool → (8,16,16,16)
→ Conv → (8,32,16,16) → Pool → (8,32,8,8)
→ Flatten → (8,2048) → Linear → (8,10)
```


### Twenty rapid shape questions

1. Linear(20,64), input (32,20) → ?
2. Embedding(10000,256), input (16,50) → ?
3. Conv2d(3,32,3,p=1), input (8,3,64,64) → ?
4. MaxPool2d(2), input (8,32,64,64) → ?
5. Flatten, input (16,64,7,7) → ?
6. Linear(128,10), input (4,12,128) → ?
7. LayerNorm(768), input (2,20,768) → ?
8. BatchNorm2d(32), input (8,32,16,16) → ?
9. AdaptiveAvgPool2d((1,1)), input (5,128,9,7) → ?
10. Conv1d(8,16,3,p=1), input (4,8,20) → ?
11. Conv2d(3,16,5), input (2,3,32,32) → ?
12. Conv2d(16,32,3,p=1,s=2), input (2,16,28,28) → ?
13. Embedding(500,32), input (6,) → ?
14. Flatten(start_dim=2), input (3,4,5,6) → ?
15. Sequential Linear 10→20→5, input (7,10) → ?
16. Dropout, input (8,12) → ?
17. ReLU, input (2,3,4) → ?
18. MaxPool2d(2), input (1,16,15,15) → ?
19. Conv2d(1,4,3,p=1), input (10,1,28,28) → ?
20. global pool + Flatten, input (9,64,11,13) → ?

<details><summary>Show answers</summary>

1. `(32,64)`
2. `(16,50,256)`
3. `(8,32,64,64)`
4. `(8,32,32,32)`
5. `(16,3136)`
6. `(4,12,10)`
7. `(2,20,768)`
8. `(8,32,16,16)`
9. `(5,128,1,1)`
10. `(4,16,20)`
11. `(2,16,28,28)`
12. `(2,32,14,14)`
13. `(6,32)`
14. `(3,4,30)`
15. `(7,5)`
16. `(8,12)`
17. `(2,3,4)`
18. `(1,16,7,7)`
19. `(10,4,28,28)`
20. `(9,64)`

</details>


### 🟡 Medium — CNN feature size

Build `features`: Conv 3→8 k3 p1, MaxPool2d(2), Conv 8→16 k3 p1, MaxPool2d(2).


In [ ]:
# Exercise: Build `features`: Conv 3→8 k3 p1, MaxPool2d(2), Conv 8→16 k3 p1, MaxPool2d(2).
features=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if features is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(features,nn.Module), "Build a module"
    assert features(torch.randn(2,3,32,32)).shape==(2,16,8,8), "Track channels and two halvings"
    print("✅ Correct!")


## 26. Common Model Patterns

```python
# Tabular multiclass
nn.Sequential(nn.Linear(num_features,128),nn.ReLU(),nn.Dropout(.2),nn.Linear(128,num_classes))

# Binary: one raw logit per sample
nn.Sequential(nn.Linear(num_features,64),nn.ReLU(),nn.Linear(64,1))

# Token IDs to learned vectors
nn.Embedding(vocab_size,hidden_size)

# CNN classifier
Conv → ReLU → Pool → Conv → ReLU → Pool → Flatten → Linear
```


### Architecture-reading exercises

For each model, identify the input size, hidden sizes, output shape for the given input, nonlinearities, and any regularization or normalization. Predict before opening the answers.

**A**
```python
nn.Sequential(
    nn.Linear(100,256),
    nn.ReLU(),
    nn.Dropout(.3),
    nn.Linear(256,10),
)
```
Input: `(32,100)`

**B**
```python
nn.Sequential(
    nn.Linear(20,64),
    nn.BatchNorm1d(64),
    nn.GELU(),
    nn.Linear(64,1),
)
```
Input: `(8,20)`

**C**
```python
nn.Sequential(
    nn.Conv2d(3,16,3,padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(16*16*16,5),
)
```
Input: `(4,3,32,32)`

<details><summary>Show answers</summary>

- **A:** 100 input features, hidden size 256, 10 logits, ReLU nonlinearity, Dropout regularization, output `(32,10)`.
- **B:** 20 input features, hidden size 64, one binary logit, BatchNorm plus GELU, output `(8,1)`.
- **C:** channels `3→16`, spatial size `32→16` after pooling, 4,096 flattened features, 5 logits, output `(4,5)`.

</details>


## 27. Common Mistakes

1. Omitting `super().__init__()` and breaking registration.
2. `Linear(20,10)` receiving last dimension 30.
3. Hard-coding the wrong CNN flatten size.
4. Calling `model.forward(x)` instead of preferred `model(x)`.
5. Applying softmax before `CrossEntropyLoss`.
6. Applying sigmoid before `BCEWithLogitsLoss`.
7. Feeding float indices to `Embedding` instead of integer IDs.
8. Supplying NHWC images when Conv2d expects NCHW.
9. Stacking Linear layers with no nonlinearity.
10. Mismatching BatchNorm's feature/channel count.
11. Forgetting dropout changes between train and eval.
12. Mismatching Conv2d input channels.


## 28. Debugging Exercises

Predict the issue before opening the answer.


### Debug 1

```python
nn.Linear(10,3)(torch.randn(32,8))
```

What is wrong?

<details><summary>Show diagnosis</summary>

The input's last dimension is 8 but the layer expects 10.

</details>


### Debug 2

```python
Embedding receives `torch.tensor([[1.,2.,3.]])`
```

What is wrong?

<details><summary>Show diagnosis</summary>

Embedding indices require an integer dtype such as torch.long.

</details>


### Debug 3

```python
Conv2d(3,16,3) receives `(8,32,32,3)`
```

What is wrong?

<details><summary>Show diagnosis</summary>

The tensor is NHWC; Conv2d expects NCHW `(8,3,32,32)`.

</details>


### Debug 4

```python
Three Linear layers with no activation
```

What is wrong?

<details><summary>Show diagnosis</summary>

The composition remains effectively linear and cannot model nonlinear boundaries.

</details>


### Debug 5

```python
Custom model omits `super().__init__()`
```

What is wrong?

<details><summary>Show diagnosis</summary>

The nn.Module base machinery is not initialized, so child/parameter registration fails.

</details>


### Debug 6

```python
CNN classifier expects `32*8*8` but features output `(N,32,7,7)`
```

What is wrong?

<details><summary>Show diagnosis</summary>

The Linear in_features is wrong. Inspect/derive feature shape or use adaptive pooling.

</details>


### Debug 7

```python
Model returns softmax then uses CrossEntropyLoss
```

What is wrong?

<details><summary>Show diagnosis</summary>

CrossEntropyLoss expects raw logits and performs log-softmax internally.

</details>


### Debug 8

```python
Model returns sigmoid then uses BCEWithLogitsLoss
```

What is wrong?

<details><summary>Show diagnosis</summary>

The loss already includes sigmoid; return raw logits.

</details>


### Debug 9

```python
BatchNorm1d(64) receives `(32,32)`
```

What is wrong?

<details><summary>Show diagnosis</summary>

The feature count is 32, not 64.

</details>


### Debug 10

```python
Dropout output changes during evaluation
```

What is wrong?

<details><summary>Show diagnosis</summary>

Ensure `model.eval()` was called; dropout should be disabled in eval mode.

</details>


### Debug 11

```python
Conv2d(3,16,3) receives `(8,1,28,28)`
```

What is wrong?

<details><summary>Show diagnosis</summary>

The layer expects 3 input channels but gets 1.

</details>


### Debug 12

```python
nn.Softmax()  # class dimension not specified
```

What is wrong?

<details><summary>Show diagnosis</summary>

Specify the dimension explicitly, typically class dimension 1 for `(batch,classes)`.

</details>


### Debug 13

```python
Embedding index equals `num_embeddings`
```

What is wrong?

<details><summary>Show diagnosis</summary>

Valid IDs are from 0 through `num_embeddings-1`; the index is out of range.

</details>


### Debug 14

```python
LayerNorm(128) receives final dimension 64
```

What is wrong?

<details><summary>Show diagnosis</summary>

normalized_shape must match the dimensions being normalized.

</details>


### Debug 15

```python
Calling `model.forward(x)` directly
```

What is wrong?

<details><summary>Show diagnosis</summary>

It may bypass Module call machinery such as hooks; call `model(x)`.

</details>


## 29. Interview Questions

Answer aloud before expanding.


### 1. What is `nn.Module`?

<details><summary>Show answer</summary>

**Short Interview Answer:** The base class for PyTorch models and layers.

**Detailed Explanation:** It provides parameter/module registration, state handling, device movement, modes, and call behavior.

</details>


### 2. Why inherit from `nn.Module`?

<details><summary>Show answer</summary>

**Short Interview Answer:** To integrate custom computation with PyTorch's model and parameter machinery.

**Detailed Explanation:** Registered children and parameters become discoverable and manageable as a model.

</details>


### 3. Why call `super().__init__()`?

<details><summary>Show answer</summary>

**Short Interview Answer:** To initialize the nn.Module base class so layers and parameters register properly.

**Detailed Explanation:** It prepares internal dictionaries and module behavior before assigning child modules.

</details>


### 4. Purpose of `forward()`?

<details><summary>Show answer</summary>

**Short Interview Answer:** It defines how inputs are transformed into outputs.

**Detailed Explanation:** Layers are usually declared in init and composed in forward.

</details>


### 5. Why `model(x)` instead of `model.forward(x)`?

<details><summary>Show answer</summary>

**Short Interview Answer:** Module call machinery invokes forward while also handling hooks and framework behavior.

**Detailed Explanation:** Direct forward calls can bypass that machinery.

</details>


### 6. What does `nn.Linear` do?

<details><summary>Show answer</summary>

**Short Interview Answer:** It applies an affine transformation to the last input dimension.

**Detailed Explanation:** For Linear(in,out), it computes xWᵀ+b and replaces the final size with out.

</details>


### 7. Shapes for `Linear(10,4)`?

<details><summary>Show answer</summary>

**Short Interview Answer:** Weight `(4,10)`, bias `(4,)`.

**Detailed Explanation:** PyTorch stores one row of weights and one bias for each output feature.

</details>


### 8. How count parameters?

<details><summary>Show answer</summary>

**Short Interview Answer:** Sum `p.numel()` over `model.parameters()`.

**Detailed Explanation:** Filter on `p.requires_grad` to count trainable parameters only.

</details>


### 9. What is a hidden layer?

<details><summary>Show answer</summary>

**Short Interview Answer:** An intermediate learned transformation between model input and output.

**Detailed Explanation:** Hidden representations are typically followed by nonlinear activations.

</details>


### 10. Why need activations?

<details><summary>Show answer</summary>

**Short Interview Answer:** They add nonlinearity so stacked layers can learn nonlinear functions.

**Detailed Explanation:** Without them, a stack of affine layers collapses to one affine mapping.

</details>


### 11. Why is ReLU common?

<details><summary>Show answer</summary>

**Short Interview Answer:** It is simple, efficient, and often yields useful sparse activations.

**Detailed Explanation:** Its zero negative slope can create dead units, which alternatives such as LeakyReLU mitigate.

</details>


### 12. What is dead ReLU?

<details><summary>Show answer</summary>

**Short Interview Answer:** A ReLU unit stuck producing zero because its input stays negative.

**Detailed Explanation:** Its gradient is then zero on that region and it may stop learning.

</details>


### 13. What is GELU?

<details><summary>Show answer</summary>

**Short Interview Answer:** A smooth activation widely used in transformers.

**Detailed Explanation:** It softly gates values rather than hard-thresholding at zero like ReLU.

</details>


### 14. What is a logit?

<details><summary>Show answer</summary>

**Short Interview Answer:** A raw unnormalized model output score.

**Detailed Explanation:** Probability transformations or suitable losses consume logits.

</details>


### 15. Logits vs probabilities?

<details><summary>Show answer</summary>

**Short Interview Answer:** Logits are unrestricted raw scores; probabilities are normalized/interpretable values.

**Detailed Explanation:** Softmax maps multiclass logits to rows summing to one; sigmoid maps binary logits to 0–1.

</details>


### 16. Why return logits during training?

<details><summary>Show answer</summary>

**Short Interview Answer:** Common losses combine activation and log operations more stably.

**Detailed Explanation:** CrossEntropyLoss and BCEWithLogitsLoss expect raw logits.

</details>


### 17. What is `nn.Sequential`?

<details><summary>Show answer</summary>

**Short Interview Answer:** A container that feeds each module's output directly to the next.

**Detailed Explanation:** It is concise for straight-line architectures.

</details>


### 18. When prefer custom Module?

<details><summary>Show answer</summary>

**Short Interview Answer:** For branching, skips, multiple inputs/outputs, or custom control flow.

**Detailed Explanation:** Forward can express arbitrary Python and reuse registered components.

</details>


### 19. What does `model.parameters()` return?

<details><summary>Show answer</summary>

**Short Interview Answer:** An iterator over registered parameter tensors.

**Detailed Explanation:** Optimizers commonly receive it to know what to update.

</details>


### 20. What does `named_parameters()` return?

<details><summary>Show answer</summary>

**Short Interview Answer:** Name-parameter pairs for registered parameters.

**Detailed Explanation:** Names help inspection, freezing, logging, and state debugging.

</details>


### 21. What does dropout do?

<details><summary>Show answer</summary>

**Short Interview Answer:** Randomly zeros and scales activations during training to regularize.

**Detailed Explanation:** It is disabled during evaluation.

</details>


### 22. Dropout train vs eval?

<details><summary>Show answer</summary>

**Short Interview Answer:** Active and random in train mode; identity-like in eval mode.

**Detailed Explanation:** Call model.eval() for deterministic inference.

</details>


### 23. What is BatchNorm?

<details><summary>Show answer</summary>

**Short Interview Answer:** Normalization using batch/channel statistics with learned scale and shift.

**Detailed Explanation:** It tracks running statistics and behaves differently in training and evaluation.

</details>


### 24. What is LayerNorm?

<details><summary>Show answer</summary>

**Short Interview Answer:** Normalization over specified feature dimensions within each sample/token.

**Detailed Explanation:** It does not depend on batch statistics and is common in sequence models.

</details>


### 25. BatchNorm vs LayerNorm?

<details><summary>Show answer</summary>

**Short Interview Answer:** BatchNorm depends on batch statistics; LayerNorm normalizes within each example/token.

**Detailed Explanation:** BatchNorm is common in CNNs; LayerNorm is common in transformers and robust to small batches.

</details>


### 26. Why LayerNorm in transformers?

<details><summary>Show answer</summary>

**Short Interview Answer:** It normalizes each token's hidden features without relying on batch size.

**Detailed Explanation:** That matches variable sequence/batch settings and transformer residual blocks.

</details>


### 27. What does Embedding do?

<details><summary>Show answer</summary>

**Short Interview Answer:** It maps integer IDs to learned dense vectors.

**Detailed Explanation:** Its weight is a lookup table shaped `(vocab,embedding_dim)`.

</details>


### 28. Embedding input dtype?

<details><summary>Show answer</summary>

**Short Interview Answer:** Usually torch.long/int64 indices.

**Detailed Explanation:** Inputs are lookup indices, not continuous feature values.

</details>


### 29. What does Conv2d do?

<details><summary>Show answer</summary>

**Short Interview Answer:** It applies learned spatial kernels across NCHW image feature maps.

**Detailed Explanation:** It changes channels and may change height/width via kernel, padding, and stride.

</details>


### 30. Common image layout?

<details><summary>Show answer</summary>

**Short Interview Answer:** NCHW: batch, channels, height, width.

**Detailed Explanation:** Conv2d interprets dimension 1 as channels.

</details>


### 31. Conv2d output size?

<details><summary>Show answer</summary>

**Short Interview Answer:** Floor((input+2p-k)/s)+1 for dilation 1.

**Detailed Explanation:** Apply independently to height and width.

</details>


### 32. What does MaxPool do?

<details><summary>Show answer</summary>

**Short Interview Answer:** It keeps local maxima while reducing spatial resolution.

**Detailed Explanation:** Batch and channel counts stay unchanged.

</details>


### 33. What does AdaptiveAvgPool do?

<details><summary>Show answer</summary>

**Short Interview Answer:** It averages spatial regions to a requested fixed output size.

**Detailed Explanation:** Global `(1,1)` pooling makes classifier input independent of original spatial resolution.

</details>


### 34. Binary vs multiclass output shapes?

<details><summary>Show answer</summary>

**Short Interview Answer:** Usually `(batch,1)` versus `(batch,num_classes)` raw logits.

**Detailed Explanation:** The matching loss interprets those logits appropriately.

</details>


### 35. Why are linear-only stacks redundant?

<details><summary>Show answer</summary>

**Short Interview Answer:** Their composition is another affine transformation.

**Detailed Explanation:** Nonlinear activations are required to expand representational power.

</details>


## 30. Knowledge Check Quiz

Choose answers before revealing the key.

1. Base class for models? A Tensor B nn.Module C Optimizer D Dataset
2. `super().__init__()` primarily: A trains B initializes Module registration C moves GPU D flattens
3. Preferred call: A model.forward(x) B model(x) C forward(model) D model.step(x)
4. Linear(20,5) weight shape: A (20,5) B (5,20) C (20,) D (5,)
5. `(32,100)` through Linear(100,10): A (100,10) B (32,100) C (32,10) D (10,32)
6. Linear(10,4) parameter count: A 40 B 44 C 14 D 50
7. Hidden nonlinearity commonly: A ReLU B Softmax always C Flatten D Embedding
8. Transformer activation commonly: A MaxPool B GELU C BatchNorm2d D Sigmoid only
9. Raw scores are: A labels B logits C gradients D kernels
10. CrossEntropyLoss normally receives: A softmax probabilities B raw logits C argmax D embeddings
11. Sequential is best for: A straight-line flow B distributed only C no modules D custom CUDA
12. Dropout in eval mode: A more random B disabled C doubles p D trains weights
13. BatchNorm eval uses: A current batch only B running statistics C no statistics D LayerNorm
14. LayerNorm common in: A transformers B image files C optimizers D labels
15. Embedding input usually: A float32 B int64 C bool D complex
16. Embedding(1000,64) weight: A (64,1000) B (1000,64) C (1000,) D (64,)
17. Conv2d expects: A NHWC B NCHW C HWNC D CHWN
18. Conv2d(3,16,3) on 28×28 no padding: A 28 B 26 C 30 D 14
19. Conv2d k3 p1 s1 spatial size: A halves B unchanged C doubles D zero
20. MaxPool2d(2) on 28×28: A 56 B 26 C 14 D 28
21. AdaptiveAvgPool2d((1,1)) changes channels? A yes B no C only training D random
22. Flatten `(16,64,7,7)`: A (16,3136) B (16,64) C (3136,) D (16,49)
23. Conv2d(3,16,3) params with bias: A 432 B 448 C 48 D 16
24. Binary output often: A (batch,1) B (batch,2,2) C scalar total D (1,batch)
25. Linear stacks without activations are: A nonlinear B effectively one affine map C convolutional D normalized

<details><summary>Show answer key</summary>

1 B · 2 B · 3 B · 4 B · 5 C · 6 B · 7 A · 8 B · 9 B · 10 B · 11 A · 12 B · 13 B · 14 A · 15 B · 16 B · 17 B · 18 B · 19 B · 20 C · 21 B · 22 A · 23 B · 24 A · 25 B

</details>


## 31. Write From Memory

Build each object without the cheat sheet.


### 🟢 Easy — Module subclass

Define and instantiate `model` as a Module that returns its input.


In [ ]:
# Exercise: Define and instantiate `model` as a Module that returns its input.
model=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if model is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(model,nn.Module) and torch.equal(model(torch.tensor([1.])),torch.tensor([1.])), "Define init and forward"
    print("✅ Correct!")


### 🟢 Easy — Linear 10→3

Create a Linear layer from 10 to 3.


In [ ]:
# Exercise: Create a Linear layer from 10 to 3.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.Linear) and (answer.in_features,answer.out_features)==(10,3), "Use nn.Linear"
    print("✅ Correct!")


### 🟢 Easy — Small MLP

Build `10 → 32 → ReLU → 2`.


In [ ]:
# Exercise: Build `10 → 32 → ReLU → 2`.
model=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if model is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert model(torch.randn(4,10)).shape==(4,2), "Check architecture"
    assert any(isinstance(m,nn.ReLU) for m in model.modules()), "Include ReLU"
    print("✅ Correct!")


### 🟢 Easy — Sequential MLP

Build the previous model specifically with Sequential.


In [ ]:
# Exercise: Build the previous model specifically with Sequential.
model=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if model is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(model,nn.Sequential) and model(torch.randn(4,10)).shape==(4,2), "Use Sequential"
    print("✅ Correct!")


### 🟢 Easy — Parameter list

Assign a list of all model parameters to `answer`.


In [ ]:
# Exercise: Assign a list of all model parameters to `answer`.
model=nn.Linear(3,2)
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,list) and len(answer)==2 and all(isinstance(p,nn.Parameter) for p in answer), "Iterate parameters"
    print("✅ Correct!")


### 🟢 Easy — Named parameters

Assign a list of `(name,param)` pairs to `answer`.


In [ ]:
# Exercise: Assign a list of `(name,param)` pairs to `answer`.
model=nn.Linear(3,2)
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert [n for n,p in answer]==['weight','bias'], "Use named_parameters"
    print("✅ Correct!")


### 🟢 Easy — Count total

Count all parameters in `model`.


In [ ]:
# Exercise: Count all parameters in `model`.
model=nn.Linear(5,4)
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert answer==24, "Sum numel over parameters"
    print("✅ Correct!")


### 🟢 Easy — Dropout

Create Dropout p=0.3.


In [ ]:
# Exercise: Create Dropout p=0.3.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.Dropout) and answer.p==.3, "Use nn.Dropout"
    print("✅ Correct!")


### 🟢 Easy — BatchNorm1d

Create BatchNorm1d for 64 features.


In [ ]:
# Exercise: Create BatchNorm1d for 64 features.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.BatchNorm1d) and answer.num_features==64, "Match features"
    print("✅ Correct!")


### 🟢 Easy — BatchNorm2d

Create BatchNorm2d for 32 channels.


In [ ]:
# Exercise: Create BatchNorm2d for 32 channels.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.BatchNorm2d) and answer.num_features==32, "Match channels"
    print("✅ Correct!")


### 🟢 Easy — LayerNorm

Create LayerNorm hidden size 768.


In [ ]:
# Exercise: Create LayerNorm hidden size 768.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.LayerNorm) and tuple(answer.normalized_shape)==(768,), "Match hidden size"
    print("✅ Correct!")


### 🟢 Easy — Embedding

Create Embedding vocabulary 50000, dimension 512.


In [ ]:
# Exercise: Create Embedding vocabulary 50000, dimension 512.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.Embedding) and answer.weight.shape==(50000,512), "Check table shape"
    print("✅ Correct!")


### 🟢 Easy — Conv2d

Create Conv2d 3→32, k3, p1.


In [ ]:
# Exercise: Create Conv2d 3→32, k3, p1.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.Conv2d) and answer(torch.randn(2,3,16,16)).shape==(2,32,16,16), "Preserve spatial size"
    print("✅ Correct!")


### 🟢 Easy — Conv1d

Create Conv1d 8→16, k3, p1.


In [ ]:
# Exercise: Create Conv1d 8→16, k3, p1.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.Conv1d) and answer(torch.randn(2,8,20)).shape==(2,16,20), "Use Conv1d"
    print("✅ Correct!")


### 🟢 Easy — Max pool

Create 2×2 max pooling.


In [ ]:
# Exercise: Create 2×2 max pooling.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.MaxPool2d) and answer(torch.randn(1,4,8,8)).shape==(1,4,4,4), "Use MaxPool2d(2)"
    print("✅ Correct!")


### 🟢 Easy — Adaptive pool

Create global adaptive average pooling.


In [ ]:
# Exercise: Create global adaptive average pooling.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.AdaptiveAvgPool2d) and answer(torch.randn(2,5,7,9)).shape==(2,5,1,1), "Output (1,1)"
    print("✅ Correct!")


### 🟢 Easy — Flatten

Create a batch-preserving flatten module.


In [ ]:
# Exercise: Create a batch-preserving flatten module.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.Flatten) and answer(torch.randn(3,2,4,5)).shape==(3,40), "Flatten from dim 1"
    print("✅ Correct!")


### 🟢 Easy — Softmax

Create Softmax across class dimension 1.


In [ ]:
# Exercise: Create Softmax across class dimension 1.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.Softmax) and answer.dim==1, "Specify dim 1"
    print("✅ Correct!")


### 🟢 Easy — Trainable count

Count trainable parameters only.


In [ ]:
# Exercise: Count trainable parameters only.
model=nn.Sequential(nn.Linear(3,4),nn.Linear(4,2))
for p in model[0].parameters():p.requires_grad=False
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert answer==10, "Filter requires_grad"
    print("✅ Correct!")


### 🟢 Easy — CNN head

Build Flatten then Linear(16*4*4,10).


In [ ]:
# Exercise: Build Flatten then Linear(16*4*4,10).
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.Module) and answer(torch.randn(2,16,4,4)).shape==(2,10), "Flatten before Linear"
    print("✅ Correct!")


### Parameter Count Challenge

Calculate all ten counts manually before running the validation cell. Include biases unless `bias=False`.

1. `Linear(128,64)`
2. `Linear(100,50) → ReLU → Linear(50,10)`
3. `Embedding(30000,256)`
4. `Conv2d(3,16,3)`
5. `Linear(64,10,bias=False)`
6. `Conv2d(16,32,3,bias=False)`
7. `Linear(20,64) → Linear(64,32) → Linear(32,5)`
8. `Embedding(1000,128)`
9. `Conv2d(1,8,5)`
10. `Conv2d(3,8,3,padding=1) → BatchNorm2d(8)`


In [ ]:
# Exercise: Fill one manual parameter count for each model above.
models_for_count=[
    nn.Linear(128,64),
    nn.Sequential(nn.Linear(100,50),nn.ReLU(),nn.Linear(50,10)),
    nn.Embedding(30000,256),
    nn.Conv2d(3,16,3),
    nn.Linear(64,10,bias=False),
    nn.Conv2d(16,32,3,bias=False),
    nn.Sequential(nn.Linear(20,64),nn.Linear(64,32),nn.Linear(32,5)),
    nn.Embedding(1000,128),
    nn.Conv2d(1,8,5),
    nn.Sequential(nn.Conv2d(3,8,3,padding=1),nn.BatchNorm2d(8)),
]
manual_counts=[None,None,None,None,None,None,None,None,None,None]
programmatic=[sum(p.numel() for p in m.parameters()) for m in models_for_count]
if any(value is None for value in manual_counts):
    print("✍️ Fill all ten manual counts, then run this validation cell.")
else:
    assert manual_counts==programmatic, f"Check each weight, bias, and normalization parameter: {list(zip(manual_counts,programmatic))}"
    print("✅ All ten parameter counts match!")


## 10-Minute Neural Network Shape Challenge

Set a timer and predict every output before opening the key.

1. `(32,784)` → Linear 784→128 → ReLU → Linear 128→10
2. `(16,3,32,32)` → Conv 3→16 p1 → Pool2 → Conv 16→32 p1 → Pool2
3. Embedding(10000,256), IDs `(8,20)`
4. Flatten `(4,32,8,8)`
5. Conv k3 p0 on `(2,3,28,28)`, out 16
6. Conv k3 p1 s2 on `(2,16,32,32)`, out 32
7. MaxPool2d(2) on `(6,64,14,14)`
8. AdaptiveAvgPool `(1,1)` on `(5,128,11,9)`
9. LayerNorm(512) on `(3,20,512)`
10. Linear(64,4) on `(2,5,64)`
11. Conv1d(8,16,k3,p1) on `(4,8,100)`
12. Binary Linear(128,1) head on `(32,128)`
13. Ten-class Linear(128,10) head on `(32,128)`
14. Global pool then Flatten on `(7,256,13,13)`
15. Two Conv2d layers with k3, p1, s1 on `(4,8,24,24)`, with output channels 16 then 32

<details><summary>Show answer key</summary>

1. `(32,10)`
2. `(16,32,8,8)`
3. `(8,20,256)`
4. `(4,2048)`
5. `(2,16,26,26)`
6. `(2,32,16,16)`
7. `(6,64,7,7)`
8. `(5,128,1,1)`
9. `(3,20,512)`
10. `(2,5,4)`
11. `(4,16,100)`
12. `(32,1)`
13. `(32,10)`
14. `(7,256)`
15. `(4,32,24,24)`

</details>


## 32. Final Model-Building Challenge

Build three interview-sized architectures. Each accepts more than one batch size.


### 🔴 Hard — Part A — Tabular classifier

Build `20→64→ReLU→Dropout(.2)→32→ReLU→5 logits` as `model_a`.


In [ ]:
# Exercise: Build `20→64→ReLU→Dropout(.2)→32→ReLU→5 logits` as `model_a`.
model_a=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if model_a is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(model_a,nn.Module), "Build a module"
    assert model_a(torch.randn(16,20)).shape==(16,5) and model_a(torch.randn(3,20)).shape==(3,5), "Preserve arbitrary batch size"
    linears=[m for m in model_a.modules() if isinstance(m,nn.Linear)]
    assert [(m.in_features,m.out_features) for m in linears]==[(20,64),(64,32),(32,5)], "Check Linear sizes"
    assert any(isinstance(m,nn.Dropout) and m.p==.2 for m in model_a.modules()), "Include Dropout(.2)"
    assert sum(isinstance(m,nn.ReLU) for m in model_a.modules())>=2, "Include two ReLUs"
    print("✅ Correct!")


### 🔴 Hard — Part B — NLP embedding classifier

Build `model_b`: Embedding(10000,128), mean over sequence, Linear(128,3).


In [ ]:
# Exercise: Build `model_b`: Embedding(10000,128), mean over sequence, Linear(128,3).
class EmbeddingClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO
    def forward(self,tokens):
        # TODO: embed, mean(dim=1), classify
        pass

model_b=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if model_b is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(model_b,nn.Module), "Instantiate EmbeddingClassifier"
    embeddings=[m for m in model_b.modules() if isinstance(m,nn.Embedding)]
    linears=[m for m in model_b.modules() if isinstance(m,nn.Linear)]
    assert len(embeddings)==1 and embeddings[0].weight.shape==(10000,128), "Check embedding table"
    assert len(linears)==1 and (linears[0].in_features,linears[0].out_features)==(128,3), "Use Linear(128,3)"
    tokens=torch.tensor([[1,2,3,4],[4,3,2,1]],dtype=torch.long)
    output=model_b(tokens)
    expected=linears[0](embeddings[0](tokens).mean(dim=1))
    assert output.shape==(2,3) and torch.allclose(output,expected), "Mean-pool embeddings across sequence dimension before classification"
    assert model_b(torch.randint(0,10000,(8,20))).shape==(8,3), "Expected one output per sequence"
    assert model_b(torch.randint(0,10000,(2,7))).shape==(2,3), "Sequence length should be flexible"
    print("✅ Correct!")


### 🔴 Hard — Part C — Small CNN

Build `model_c`: Conv 3→16 p1, ReLU, Pool; Conv 16→32 p1, ReLU, Pool; Flatten; Linear 2048→64, ReLU, Linear 64→10.


In [ ]:
# Exercise: Build `model_c`: Conv 3→16 p1, ReLU, Pool; Conv 16→32 p1, ReLU, Pool; Flatten; Linear 2048→64, ReLU, Linear 64→10.
model_c=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if model_c is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(model_c,nn.Module), "Build a module"
    assert model_c(torch.randn(1,3,32,32)).shape==(1,10), "Check batch size 1"
    assert model_c(torch.randn(5,3,32,32)).shape==(5,10), "Check arbitrary batch"
    convs=[m for m in model_c.modules() if isinstance(m,nn.Conv2d)]
    linears=[m for m in model_c.modules() if isinstance(m,nn.Linear)]
    pools=[m for m in model_c.modules() if isinstance(m,nn.MaxPool2d)]
    assert [(m.in_channels,m.out_channels,m.kernel_size,m.padding) for m in convs]==[(3,16,(3,3),(1,1)),(16,32,(3,3),(1,1))], "Check convolution channels, kernels, and padding"
    assert len(pools)==2 and all(m.kernel_size==2 for m in pools), "Include two 2x2 max-pooling layers"
    assert sum(isinstance(m,nn.ReLU) for m in model_c.modules())>=3, "Include ReLU after both convolutions and the hidden Linear layer"
    assert any(isinstance(m,nn.Flatten) for m in model_c.modules()), "Flatten before the classifier"
    assert [(m.in_features,m.out_features) for m in linears]==[(2048,64),(64,10)], "Check classifier dimensions"
    print("✅ Correct!")


## 33. Neural Network Cheat Sheet

```python
import torch.nn as nn

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc=nn.Linear(10,2)
    def forward(self,x):
        return self.fc(x)

nn.Linear(in_features,out_features)
nn.ReLU(); nn.LeakyReLU(); nn.GELU(); nn.Sigmoid(); nn.Tanh(); nn.Softmax(dim=1)
nn.Sequential(nn.Linear(10,32),nn.ReLU(),nn.Linear(32,2))
nn.Dropout(p=.5)
nn.BatchNorm1d(64); nn.BatchNorm2d(32); nn.LayerNorm(768)
nn.Flatten()
nn.Embedding(num_embeddings=50000,embedding_dim=768)
nn.Conv1d(8,16,kernel_size=3,padding=1)
nn.Conv2d(in_channels=3,out_channels=32,kernel_size=3,padding=1)
nn.MaxPool2d(2); nn.AdaptiveAvgPool2d((1,1))
model.parameters(); model.named_parameters()
sum(p.numel() for p in model.parameters())
sum(p.numel() for p in model.parameters() if p.requires_grad)
```

```text
Model = Layers + Parameters + forward()
Input → Linear/Conv/Embedding → Activation → more layers → raw logits
```


### Interview quick reference

| Term | Meaning |
|---|---|
| `nn.Module` | base class for PyTorch models |
| `forward()` | defines computation |
| `nn.Linear` | fully connected affine transformation |
| ReLU | common hidden activation |
| GELU | common transformer activation |
| Dropout | regularization active in train mode |
| BatchNorm | batch-based normalization, common in CNNs |
| LayerNorm | within-feature normalization, common in transformers |
| Embedding | integer IDs → learned vectors |
| Conv2d | image feature extraction |
| Logits | raw model output scores |


## Before Moving to `05_losses_optimizers.ipynb`

- [ ] I can create a model using `nn.Module`.
- [ ] I know why `super().__init__()` is required.
- [ ] I understand `forward()` and why `model(x)` is preferred.
- [ ] I can use `nn.Linear` and predict its output/parameter shapes.
- [ ] I can calculate Linear and Conv parameter counts.
- [ ] I understand why activation functions are needed.
- [ ] I know the main uses of ReLU, GELU, Sigmoid, and Softmax.
- [ ] I understand logits and binary vs multiclass outputs.
- [ ] I can create models using `nn.Sequential`.
- [ ] I can inspect and count model parameters.
- [ ] I understand Dropout, BatchNorm, and LayerNorm.
- [ ] I can explain BatchNorm vs LayerNorm.
- [ ] I can use `nn.Embedding` and predict its shapes.
- [ ] I understand basic Conv2d syntax and pooling shapes.
- [ ] I can build a simple MLP, embedding classifier, and CNN.
